In [ ]:
import pandas as pd
# llamaremos el archivo de excel que se encuentra en la siguiente ruta data\raw\Online Retail.xlsx

df = pd.read_excel('../data/raw/Online Retail.xlsx', sheet_name='Online Retail')

In [ ]:
print(df.head())
print(df.info())

In [ ]:
print(df.isnull().sum())
print(df.describe())    

In [ ]:
df[df["Quantity"] < 0].head(20)

In [ ]:
df.describe()

In [ ]:
df[df["Quantity"] < 0]["InvoiceNo"].head(20)

In [ ]:
#Vamos a medir cuántos registros negativos existen
print((df["Quantity"] < 0).sum())

In [ ]:
df[df["InvoiceNo"].astype(str).str.startswith("C")].shape

#ahora mostramos el número de registros negativos que tienen un InvoiceNo que empieza con C

In [ ]:
df[
    (df["Quantity"] < 0) &
    (df["InvoiceNo"].astype(str).str.startswith("C"))
].shape

In [ ]:
df["Country"].value_counts().head(10)

In [ ]:
df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

In [ ]:
# 1. ¿Cuántas filas duplicadas existen?
df.duplicated().sum()

In [ ]:
# 2. ¿Cuántos InvoiceNo diferentes existen?
df["InvoiceNo"].nunique()

In [ ]:
# 3. ¿Cuántos CustomerID diferentes existen?
df["CustomerID"].nunique()

In [ ]:
# 4. ¿Cuántos productos diferentes existen?
df["StockCode"].nunique()

In [ ]:
# 5. ¿Cuántos países diferentes existen?
df["Country"].nunique()

In [ ]:
# crearemos la columna TotalPrice que es el resultado de multiplicar Quantity por UnitPrice
df["Revenue"] = df["Quantity"] * df["UnitPrice"] 

In [ ]:
df[["Quantity", "UnitPrice", "Revenue"]].head(10)

In [ ]:
# crearemos una table que agrupe por stockcode y contenga las columnas description, Quantity, Revenue, y el numero de facturas diferentes que contienen ese producto. Para esto usaremos la función agg de pandas. y que este en orden descendente por Revenue.
df.groupby("StockCode").agg({
    "Description": "first",
    "Quantity": "sum",
    "Revenue": "sum",
    "InvoiceNo": "nunique"
}).sort_values("Revenue", ascending=False)  

In [ ]:
'''Decisión #1

Objetivo:
Analizar únicamente productos vendidos.

Regla aplicada:
Excluir registros con Quantity < 0 y Revenue < 0.

Justificación:
Estos registros probablemente representan devoluciones o cancelaciones y podrían distorsionar el análisis de ventas por producto.

Limitaciones:
La clasificación se realiza únicamente con la información disponible. Si en el futuro se dispone de un catálogo de tipos de transacción, la regla deberá revisarse.'''

In [ ]:
df_productos = df.groupby("StockCode").agg({
    "Description": "first",
    "Quantity": "sum",
    "Revenue": "sum",
    "InvoiceNo": "nunique"
})

In [ ]:
# ahore crearemos una nueva columna llamada participacion que es el resultado de dividir la columna Revenue entre la suma total de Revenue. Esto nos dará la participación de cada producto en el total de ventas.
df_productos["Participacion"] = df_productos["Revenue"]*100 / df_productos["Revenue"].sum()

df_productos.sort_values("Participacion", ascending=False).head(20)

In [ ]:
# adicionaremo una columna de participacion acumulada que es el resultado de sumar la columna participacion de manera acumulativa. Esto nos dará la participación acumulada de cada producto en el total de ventas.
df_productos["Participacion_Acumulada"] = df_productos.sort_values("Participacion", ascending=False)["Participacion"].cumsum()
df_productos.sort_values("Participacion", ascending=False).head(20)

In [ ]:
# ahora sumaremos la cantidad de stockcodes que representan el 80% de las ventas. Para esto usaremos la función cumsum de pandas y luego contaremos el número de stockcodes que tienen una participación acumulada menor o igual a 80.
df_productos[df_productos["Participacion_Acumulada"] <= 80].shape

In [ ]:
print(df_productos.head())
print(df_productos.info())

In [ ]:
top20 = (
    df_productos
    .sort_values("Revenue", ascending=False)
    .head(20)
)

In [ ]:
# graficaremos los 20 productos con mayor participación en las ventas. Para esto usaremos la librería matplotlib y la función bar de pandas.
import matplotlib.pyplot as plt

In [ ]:
#generamos un gráfico de barras horizontales con los 20 productos con mayor ventas teniendo como eje x la descripción de los productos. 
#Para esto usaremos la función barh de pandas y la función plot de matplotlib. Este debe contener un título y etiquetas en los ejes x e y.
#que contenga plt.tight_layout() y plt.grid()
#el eje y invertido para que el producto con mayor ventas esté en la parte superior del gráfico.
#cada barra tendrá su etiqueta con el valor de ventas correspondiente


fig, ax = plt.subplots(figsize=(10, 8))

top20.plot.barh(
    x="Description",
    y="Revenue",
    ax=ax,
    legend=False
)

ax.set_title("Top 20 productos por facturación")
ax.set_xlabel("Facturación (£)")
ax.set_ylabel("Descripción del Producto")

ax.bar_label(ax.containers[0])

ax.invert_yaxis()

ax.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()




## Hallazgo 1. Concentración de la facturación

El análisis muestra que los 20 productos con mayor facturación representan aproximadamente el 14 % de los ingresos del negocio. Aunque el catálogo cuenta con más de 4.000 referencias, un grupo reducido de productos concentra una parte importante de la facturación.

Este resultado sugiere que estos productos constituyen activos estratégicos para la empresa y deben ser considerados prioritarios en decisiones relacionadas con inventario, disponibilidad y campañas comerciales.

In [ ]:
#crearemos un datafreme llamado pareto  que unicamente que contenga la participación acumulada de cada producto en forma descendente


pareto = (
    df_productos
    .sort_values("Revenue", ascending=False)
    [["Participacion_Acumulada"]]
    .reset_index(drop=True)
)

pareto.index = pareto.index + 1
pareto.index.name = "Ranking"



In [ ]:
#graficaremos pareto con un gráfico de lineas en donde el eje x sea el índice y el eje y sea la participación acumulada. 
#agregaremos una linea que corte en el eje x en el 80% de participación acumulada y otra linea que corte en el eje y en el 80% de participación acumulada.

fig, ax = plt.subplots(figsize=(10, 8))

pareto.plot(
    y="Participacion_Acumulada",
    ax=ax,
    legend=False
)
ax.set_title("Curva de Pareto de productos por facturación")
ax.set_xlabel("Ranking de productos")
ax.set_ylabel("Participación Acumulada (%)")

# Agregar líneas de corte y etuqueta de linea de corte en el eje x en el 80% de participación acumulada

ax.axhline(y=80, color='r', linestyle='--', label='Umbral de pareto (80%')
ax.axvline(x=pareto[pareto["Participacion_Acumulada"] <= 80].shape[0], color='g', linestyle='--', label='736 productos representan el 80% de la facturacion')
ax.annotate(
    '736 productos representan el 80% de la facturación',
    xy=(pareto[pareto["Participacion_Acumulada"] <= 80].shape[0], 80),
    xytext=(5, 10),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.1')
)



ax.legend()
plt.show()


In [ ]:
#graficaremos pareto con un gráfico de lineas en donde el eje x sea el índice y el eje y sea la participación acumulada.

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(pareto.index, pareto['Participacion_Acumulada'], marker='o', color='C0')
ax.set_xlabel('Ranking')
ax.set_ylabel('Participación Acumulada (%)')
ax.set_title('Diagrama de Pareto de Ventas')
plt.tight_layout()
plt.show()